# Create Agent

### Getting Started

This sample demonstrates how to evaluate Azure AI Agent
Before running the sample:
```bash
pip install azure-ai-projects azure-identity azure-ai-evaluation
```
Set these environment variables with your own values:
1) **PROJECT_CONNECTION_STRING** - The project connection string, as found in the overview page of your Azure AI Foundry project.
2) **MODEL_DEPLOYMENT_NAME** - The deployment name of the AI model, as found under the "Name" column in the "Models + endpoints" tab in your Azure AI Foundry project.
3) **AZURE_OPENAI_ENDPOINT** - Azure Open AI Endpoint to be used for evaluation.
4) **AZURE_OPENAI_API_KEY** - Azure Open AI Key to be used for evaluation.
5) **AZURE_OPENAI_API_VERSION** - Azure Open AI Api version to be used for evaluation.
6) **AZURE_SUBSCRIPTION_ID** - Azure Subscription Id of Azure AI Project
7) **PROJECT_NAME** - Azure AI Project Name
8) **RESOURCE_GROUP_NAME** - Azure AI Project Resource Group Name


In [1]:
import os, json
import pandas as pd
import time
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from user_functions import user_functions
from dotenv import load_dotenv

load_dotenv()

from azure.ai.projects import __version__ as projects_version
from packaging.version import Version
# some dependencies have been updated with breaking changes -- indicates whether to use updated models and APIs or not
updated_agents = Version(projects_version) > Version("1.0.0b10") or projects_version.startswith("1.0.0a")

if updated_agents:
    from azure.ai.agents.models import FunctionTool, ToolSet
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
        api_version="v1"
        #api_version=os.environ.get("AZURE_OPENAI_API_VERSION")
    )
else:
    from azure.ai.projects.models import FunctionTool, ToolSet
    project_client = AIProjectClient.from_connection_string(
        credential=DefaultAzureCredential(),
        conn_str=os.environ["PROJECT_CONNECTION_STRING"],
    )

AGENT_NAME = "Seattle Tourist Assistant PrP"

# Adding Tools to be used by Agent 
functions = FunctionTool(user_functions)

toolset = ToolSet()
toolset.add(functions)

### Create Agent

In [2]:
agent = project_client.agents.create_agent(
    model=os.environ["MODEL_DEPLOYMENT_NAME"],
    name=AGENT_NAME,
    instructions="You are a helpful assistant",
    toolset=toolset
)

print(f"Created agent, ID: {agent.id}")

Created agent, ID: asst_hleiwxx3KoL2FAKW3RmP0qlT


### Create Thread

In [3]:
if updated_agents:
    thread = project_client.agents.threads.create()
else:
    thread = project_client.agents.create_thread()
print(f"Created thread, ID: {thread.id}")

Created thread, ID: thread_IlgiNKXRWZHcwPK8DWkvnOb7


## Conversation with Agent
Use below cells to have conversation with the agent
- `Create Message[1]`
- `Execute[2]`

### Create Message[1]

In [4]:
# Create message to thread

MESSAGE = "Can you send me an email with weather information for Seattle?"

if updated_agents:
    message = project_client.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content=MESSAGE,
    )        
else:
    message = project_client.agents.create_message(
        thread_id=thread.id,
        role="user",
        content=MESSAGE,
    )
    
print(f"Created message, ID: {message.id}")

Created message, ID: msg_ftDrxyFBePjZYpeLGqUaqB0Z


### Execute[2]

In [5]:

if updated_agents:
    from azure.ai.agents.models import (
        FunctionTool,
        ListSortOrder,
        RequiredFunctionToolCall,
        SubmitToolOutputsAction,
        ToolOutput,
    )
    run = project_client.agents.runs.create(thread_id=thread.id, agent_id=agent.id)
    
    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(1)
        run = project_client.agents.runs.get(thread_id=thread.id, run_id=run.id)

        if run.status == "requires_action" and isinstance(run.required_action, SubmitToolOutputsAction):
            tool_calls = run.required_action.submit_tool_outputs.tool_calls
            if not tool_calls:
                print("No tool calls provided - cancelling run")
                project_client.agents.runs.cancel(thread_id=thread.id, run_id=run.id)
                break

            tool_outputs = []
            for tool_call in tool_calls:
                if isinstance(tool_call, RequiredFunctionToolCall):
                    try:
                        print(f"Executing tool call: {tool_call}")
                        output = functions.execute(tool_call)
                        tool_outputs.append(
                            ToolOutput(
                                tool_call_id=tool_call.id,
                                output=output,
                            )
                        )
                    except Exception as e:
                        print(f"Error executing tool_call {tool_call.id}: {e}")

            print(f"Tool outputs: {tool_outputs}")
            if tool_outputs:
                project_client.agents.runs.submit_tool_outputs(thread_id=thread.id, run_id=run.id, tool_outputs=tool_outputs)
    print(f"Run status: {run.status}")

else:
    from azure.ai.projects.models import (
        FunctionTool,
        ListSortOrder,
        RequiredFunctionToolCall,
        SubmitToolOutputsAction,
        ToolOutput,
    )
    run = project_client.agents.create_run(thread_id=thread.id, agent_id=agent.id)
    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(1)
        run = project_client.agents.get_run(thread_id=thread.id, run_id=run.id)

        if run.status == "requires_action" and isinstance(run.required_action, SubmitToolOutputsAction):
            tool_calls = run.required_action.submit_tool_outputs.tool_calls
            if not tool_calls:
                print("No tool calls provided - cancelling run")
                project_client.agents.cancel_run(thread_id=thread.id, run_id=run.id)
                break

            tool_outputs = []
            for tool_call in tool_calls:
                if isinstance(tool_call, RequiredFunctionToolCall):
                    try:
                        print(f"Executing tool call: {tool_call}")
                        output = functions.execute(tool_call)
                        tool_outputs.append(
                            ToolOutput(
                                tool_call_id=tool_call.id,
                                output=output,
                            )
                        )
                    except Exception as e:
                        print(f"Error executing tool_call {tool_call.id}: {e}")

            print(f"Tool outputs: {tool_outputs}")
            if tool_outputs:
                project_client.agents.submit_tool_outputs_to_run(thread_id=thread.id, run_id=run.id, tool_outputs=tool_outputs)
    print(f"Run status: {run.status}")


print(f"Run finished with status: {run.status}")

if run.status == "failed":
    print(f"Run failed: {run.last_error}")

print(f"Run ID: {run.id}")

Executing tool call: {'id': 'call_ctmlqsSnrbgL2BStEAhNRm0w', 'type': 'function', 'function': {'name': 'fetch_weather', 'arguments': '{"location":"Seattle"}'}}
Tool outputs: [{'tool_call_id': 'call_ctmlqsSnrbgL2BStEAhNRm0w', 'output': '{"weather": "Rainy, 14\\u00b0C"}'}]
Executing tool call: {'id': 'call_Rg57KQw3cbaQ7I37ljpjQnCl', 'type': 'function', 'function': {'name': 'send_email', 'arguments': '{"recipient":"user@example.com","subject":"Weather Information for Seattle","body":"The current weather in Seattle is rainy with a temperature of 14°C. If you need this information in Fahrenheit or any other details, please let me know."}'}}
Sending email to user@example.com...
Subject: Weather Information for Seattle
Body:
The current weather in Seattle is rainy with a temperature of 14°C. If you need this information in Fahrenheit or any other details, please let me know.
Tool outputs: [{'tool_call_id': 'call_Rg57KQw3cbaQ7I37ljpjQnCl', 'output': '{"message": "Email successfully sent to user

### List Messages

In [6]:
if updated_agents:
    for message in project_client.agents.messages.list(thread.id, order="asc"):
        print(f"Role: {message.role}")
        print(f"Content: {message.content[0].text.value}")
        print("-" * 40)
else:
    for message in project_client.agents.list_messages(thread.id, order="asc").data:
        print(f"Role: {message.role}")
        print(f"Content: {message.content[0].text.value}")
        print("-" * 40)

Role: MessageRole.USER
Content: Can you send me an email with weather information for Seattle?
----------------------------------------
Role: MessageRole.AGENT
Content: The weather in Seattle is currently rainy with a temperature of 14°C. I have sent this information to your email. Let me know if there's anything else you'd like!
----------------------------------------


# Evaluate

### Get data from agent

In [7]:
import json
from azure.ai.evaluation import AIAgentConverter

# Initialize the converter that will be backed by the project.
converter = AIAgentConverter(project_client)

thread_id = thread.id
run_id = run.id

converted_data = converter.convert(thread_id=thread_id, run_id=run_id)
print(json.dumps(converted_data, indent=4))

Class AIAgentConverter: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class FDPAgentDataRetriever: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AIAgentDataRetriever: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


{
    "query": [
        {
            "role": "system",
            "content": "You are a helpful assistant"
        },
        {
            "createdAt": "2025-11-10T11:48:08Z",
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Can you send me an email with weather information for Seattle?"
                }
            ]
        }
    ],
    "response": [
        {
            "createdAt": "2025-11-10T11:48:12Z",
            "run_id": "run_mvR3OYdMWrcxfDBSjUzYTT6d",
            "role": "assistant",
            "content": [
                {
                    "type": "tool_call",
                    "tool_call_id": "call_ctmlqsSnrbgL2BStEAhNRm0w",
                    "name": "fetch_weather",
                    "arguments": {
                        "location": "Seattle"
                    }
                }
            ]
        },
        {
            "createdAt": "2025-11-10T11:48:14Z",
    

In [8]:
# Save the converted data to a JSONL file

file_name = "evaluation_data.jsonl"
evaluation_data = converter.prepare_evaluation_data(thread_ids=thread.id, filename=file_name)


In [9]:
load_dotenv()

True

### Setting up evaluator

In [10]:
from azure.ai.evaluation import ToolCallAccuracyEvaluator , AzureOpenAIModelConfiguration, IntentResolutionEvaluator, TaskAdherenceEvaluator
from pprint import pprint

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    #api_version='2025-04-01-preview',
    azure_deployment=os.environ["MODEL_DEPLOYMENT_NAME"],
)
# Needed to use content safety evaluators
azure_ai_project={
    "subscription_id": os.environ["AZURE_SUBSCRIPTION_ID"],
    "project_name": os.environ["PROJECT_NAME"],
    "resource_group_name": os.environ["RESOURCE_GROUP_NAME"],
}

tool_call_accuracy = ToolCallAccuracyEvaluator(model_config=model_config)
intent_resolution = IntentResolutionEvaluator(model_config=model_config)
task_adherence = TaskAdherenceEvaluator(model_config=model_config)

Class ToolCallAccuracyEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntentResolutionEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class TaskAdherenceEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [11]:
tool_call_accuracy(query=converted_data['query'], response=converted_data['response'], tool_definitions=converted_data['tool_definitions'])

{'tool_call_accuracy': 5.0,
 'gpt_tool_call_accuracy': 5.0,
 'tool_call_accuracy_result': 'pass',
 'tool_call_accuracy_threshold': 3,
 'tool_call_accuracy_reason': "Let's think step by step: The user's last query was to send weather information for Seattle via email. The agent correctly identified the need to fetch weather data and then send an email. The tool calls made include 'fetch_weather' with 'Seattle' as the parameter, which is correct and grounded from the conversation. The second call is 'send_email' with appropriate recipient, subject, and body, which is also correct and relevant. Both calls used the correct parameters, and the sequence is logical. No errors or unnecessary calls are evident. The tools used are appropriate, and the parameters are properly grounded, fulfilling the user's request efficiently and accurately.",
 'tool_call_accuracy_details': {'tool_calls_made_by_agent': 2,
  'correct_tool_calls_made_by_agent': 2,
  'per_tool_call_details': [{'tool_name': 'fetch_w

### Run Evaluator

In [ ]:
from azure.ai.evaluation import evaluate

response = evaluate(
    data=file_name,
    evaluators={
        "tool_call_accuracy": tool_call_accuracy,
        "intent_resolution": intent_resolution,
        "task_adherence": task_adherence
    },
    azure_ai_project=os.environ["PROJECT_ENDPOINT"]
)
pprint(f'AI Foundary URL: {response.get("studio_url")}')

2025-11-10 12:48:32 +0100 6257422336 execution.bulk     INFO     Finished 1 / 3 lines.
2025-11-10 12:48:32 +0100 6257422336 execution.bulk     INFO     Average execution time for completed lines: 0.94 seconds. Estimated time for incomplete lines: 1.88 seconds.
2025-11-10 12:48:32 +0100 6257422336 execution.bulk     INFO     Finished 2 / 3 lines.
2025-11-10 12:48:32 +0100 6257422336 execution.bulk     INFO     Average execution time for completed lines: 0.48 seconds. Estimated time for incomplete lines: 0.48 seconds.
2025-11-10 12:48:32 +0100 6257422336 execution.bulk     INFO     Finished 3 / 3 lines.
2025-11-10 12:48:32 +0100 6257422336 execution.bulk     INFO     Average execution time for completed lines: 0.34 seconds. Estimated time for incomplete lines: 0.0 seconds.
2025-11-10 12:48:32 +0100 6274248704 execution.bulk     INFO     Finished 1 / 3 lines.
2025-11-10 12:48:32 +0100 6274248704 execution.bulk     INFO     Average execution time for completed lines: 1.02 seconds. Estimate

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "tool_call_accuracy_20251110_114831_204236"
Run status: "Completed"
Start time: "2025-11-10 11:48:31.204236+00:00"
Duration: "0:00:03.004654"

======= Combined Run Summary (Per Evaluator) =======

{
    "tool_call_accuracy": {
        "status": "Completed",
        "duration": "0:00:03.004654",
        "completed_lines": 3,
        "failed_lines": 0,
        "log_path": null,
        "error_message": null,
        "error_code": null
    },
    "intent_resolution": {
        "status": "Completed",
        "duration": "0:00:02.002903",
        "completed_lines": 3,
        "failed_lines": 0,
        "log_path": null,
        "error_message": null,
        "error_code": null
    },
    "task_adherence": {
        "status": "Completed",
        "duration": "0:00:02.004758",
        "completed_lines": 3,
        "failed_lines": 0,
        "log_path": null,
        "error_message": null,
        "error_code": null
    }
}


'AI Foundary URL: None'


{'metrics': {'intent_resolution.binary_aggregate': 1.0,
             'intent_resolution.gpt_intent_resolution': 5.0,
             'intent_resolution.intent_resolution': 5.0,
             'intent_resolution.intent_resolution_completion_tokens': 48.0,
             'intent_resolution.intent_resolution_prompt_tokens': 1923.6666666666667,
             'intent_resolution.intent_resolution_threshold': 3.0,
             'intent_resolution.intent_resolution_total_tokens': 1971.6666666666667,
             'task_adherence.binary_aggregate': 1.0,
             'task_adherence.task_adherence': 1.0,
             'tool_call_accuracy.binary_aggregate': 1.0,
             'tool_call_accuracy.gpt_tool_call_accuracy': 5.0,
             'tool_call_accuracy.tool_call_accuracy': 5.0,
             'tool_call_accuracy.tool_call_accuracy_completion_tokens': 378.6666666666667,
             'tool_call_accuracy.tool_call_accuracy_prompt_tokens': 3027.3333333333335,
             'tool_call_accuracy.tool_call_accurac

In [13]:
pprint(response)